# Rolling SLA portfolio MVP

This notebook demonstrates the same decision engine used by the localhost application.
It answers a deliberately narrow question:

> Given accepted SLAs, a candidate SLA pool, current resources, named uncertainty
> scenarios, launch opportunities and candidate payloads, which manifest protects existing
> commitments and has positive expected commercial value?

All times are seconds relative to the exact UTC `as_of` timestamp. The planner uses an
exact binary manifest MILP and runs MDLS routes in every named uncertainty scenario.
Routes use a time-dependent Edelbaum/J2 transfer table built for the problem orbits.

In [ ]:
from copy import deepcopy
import json
from pathlib import Path

import matplotlib.pyplot as plt

from SLA.mvp.planner import evaluate_portfolio

repo = Path.cwd()
if repo.name == "research":
    repo = repo.parent.parent
elif repo.name == "SLA":
    repo = repo.parent

example_path = repo / "SLA" / "mvp" / "example_problem.json"
problem = json.loads(example_path.read_text(encoding="utf-8"))
print(f"Loaded {problem['problem_id']} as of {problem['as_of']}")

## 1. Inspect the decision state

The example has one protected commitment, one candidate contract and three named
scenarios. The optimizer constructs a manifest from individual commodity and spacecraft
payload candidates while enforcing launch mass capacity.

In [ ]:
print("Accepted SLAs:")
for sla in problem["accepted_slas"]:
    print(f"  {sla['sla_id']}: deadline={sla['deadline_s']:,.0f} s, required reliability={sla['required_reliability']:.0%}")

print("\nCandidate SLAs:")
for sla in problem["candidate_slas"]:
    print(f"  {sla['sla_id']}: revenue={sla['revenue']:,.0f}, required reliability={sla['required_reliability']:.0%}")

print("\nManifest candidates:")
for item in problem["manifest_candidates"]:
    print(f"  {item['item_id']}: mass={item['mass_kg']:,.0f} kg, cost={item['cost']:,.0f}, launch={item['launch_event']}")

print("\nLaunch opportunities:")
for launch in problem["launch_opportunities"]:
    print(f"  {launch['launch_event']}: capacity={launch['capacity_kg']:,.0f} kg, launch={launch['planned_launch_s']:,.0f} s")

print("\nUncertainty scenarios:")
for scenario in problem["scenarios"]:
    print(f"  {scenario['scenario_id']}: probability={scenario['probability']:.0%}")

## 2. Evaluate the portfolio

The baseline contains only accepted SLAs. The manifest MILP enumerates feasible binary
payload combinations and evaluates each with the candidate pool added. A manifest is
admissible only when every SLA meets its required scenario-weighted reliability.

In [ ]:
result = evaluate_portfolio(problem)
print(result["recommendation"])
print(result["explanation"])
print(f"Baseline protects accepted SLAs: {result['baseline']['reliable']}")

In [ ]:
header = ("Manifest", "Reliable", "Expected revenue", "Manifest cost", "Expected penalty", "Net value")
rows = []
for option in result["options"]:
    rows.append((
        option["package_name"],
        "yes" if option["reliable"] else "no",
        option["expected_revenue"],
        option["package_cost"],
        option["expected_penalty"],
        option["expected_net_value"],
    ))

print(f"{header[0]:32s} {header[1]:>9s} {header[2]:>17s} {header[3]:>14s} {header[4]:>18s} {header[5]:>14s}")
for row in rows:
    print(f"{row[0]:32s} {row[1]:>9s} {row[2]:17,.0f} {row[3]:14,.0f} {row[4]:18,.0f} {row[5]:14,.0f}")

In [ ]:
names = [row[0] for row in rows]
net_values = [row[-1] / 1e6 for row in rows]
colors = ["tab:green" if row[1] == "yes" else "tab:red" for row in rows]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(names, net_values, color=colors, alpha=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Expected net value [million currency units]")
ax.set_title("Candidate manifests: green options meet all reliability requirements")
plt.tight_layout()
plt.show()

## 3. Explain the selected plan

Reliability is the sum of probabilities of scenarios in which an SLA can be served.
The selected manifest remains acceptable only while that reliability exceeds the
contractual threshold.

In [ ]:
selected = next(option for option in result["options"]
                if option["package_id"] == result["selected_package_id"])

for sla_id, reliability in selected["reliability"].items():
    required = selected["requirements"][sla_id]
    print(f"{sla_id}: estimated={reliability:.1%}, required={required:.1%}, pass={reliability >= required}")

print("\nFailure drivers:")
for driver in selected["failure_drivers"]:
    print(f"  {driver['reason']}: {driver['probability_weight']:.1%} probability weight")

## 4. Sensitivity to launch failure

This experiment moves probability mass between the nominal and launch-failure
scenarios. It shows the exact launch-risk level at which the candidate's required
reliability is no longer met.

In [ ]:
failure_probabilities = [x / 100 for x in range(0, 31, 2)]
candidate_reliability = []
net_value = []
admissible = []

for failure_probability in failure_probabilities:
    trial = deepcopy(problem)
    # Hold the 20% servicer-outage scenario fixed and move the remaining 80%
    # between nominal operation and launch failure.
    trial["scenarios"][0]["probability"] = 0.8 - failure_probability
    trial["scenarios"][1]["probability"] = failure_probability
    trial["scenarios"][2]["probability"] = 0.2
    trial_result = evaluate_portfolio(trial)
    option = next(x for x in trial_result["options"] if x["package_id"] == selected["package_id"])
    candidate_reliability.append(option["reliability"]["CANDIDATE-REFUEL-02"])
    net_value.append(option["expected_net_value"] / 1e6)
    admissible.append(option["reliable"])

required = problem["candidate_slas"][0]["required_reliability"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(failure_probabilities, candidate_reliability, marker="o")
axes[0].axhline(required, color="tab:red", linestyle="--", label=f"Required {required:.0%}")
axes[0].set(xlabel="Launch-failure probability", ylabel="Candidate reliability", title="Contract reliability")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(failure_probabilities, net_value, marker="o", color="tab:green")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set(xlabel="Launch-failure probability", ylabel="Expected net value [million]", title="Commercial value")
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

first_failure = next((p for p, ok in zip(failure_probabilities, admissible) if not ok), None)
print("First tested launch-failure probability that makes the package inadmissible:", first_failure)

## 5. Current implementation and fidelity upgrades

The complete algorithmic loop is now present:

1. A binary manifest MILP selects deployment and resupply payloads subject to launch capacity.
2. MDLS produces operational routes for every manifest/scenario evaluation.
3. Commodity balances determine whether routed services have inventory.
4. Scenario weights determine SLA reliability and expected commercial value.

The transfer layer now builds a time-dependent Edelbaum/J2 table for the relevant
orbits, departure epochs and flight times. Explicit inventory pickup visits inside
MDLS routes and a production MILP backend are later scaling upgrades.

The acceptance result remains explainable: selected manifest, reliability by SLA,
incremental value, launch dependencies and scenario-specific failure reasons.